# Deep-Dive Conceptual Roadmap & Dataset Ecosystem

## Technical Terminology & Mechanics

**BitFit (Bias-term Fine-Tuning)** is an extreme, sparse reparameterization methodology designed for Parameter-Efficient Fine-Tuning (PEFT). Instead of introducing auxiliary adapter weights (like LoRA) or updating the entire network architecture, BitFit isolates and updates **only the existing bias tensors** within the model's linear and normalization layers.

### Definition & Mechanics

BitFit freezes the core multi-dimensional weight matrices of a neural network and computes gradient updates exclusively for the **1D bias vectors** (e.g., $b$ in the affine transformation $y = Wx + b$), fundamentally shifting the activation threshold of neurons without altering their learned feature extraction patterns.

### The Engineering Problem Solved

Full-parameter fine-tuning introduces catastrophic VRAM bottlenecks due to:

- Massive **optimizer states** (e.g., AdamW requires 8 bytes per trainable parameter)
- **Gradient checkpointing** costs

BitFit solves this by drastically reducing the trainable parameter count to typically **less than 0.1% of the model**, allowing for rapid domain adaptation on consumer-grade hardware while heavily mitigating catastrophic forgetting.

---

## The Human Element: Dataset Ecosystem

For BitFit to be effective in generative LLMs, the base model must already possess the underlying world knowledge, as updating bias terms **cannot inject substantial new facts**. Therefore, BitFit is highly effective for **format alignment** and **style transfer**.

1. `timdettmers/openassistant-guanaco`

    **Why it's used:** This dataset contains highly structured, multilingual, human-annotated conversations. Because BitFit merely shifts activation thresholds, using a high-quality conversational dataset aligns the model's existing knowledge into a **dialogue format** without destroying its pre-trained semantic space.

2. `HuggingFaceH4/no_robots`

    **Why it's used:** A high-quality Instruction Tuning dataset comprising **10,000 instructions and demonstrations** created by expert human annotators. BitFit excels here because the strict instruction-response structure requires minimal re-routing of the network's internal logic — it simply needs **"nudging" via bias offsets** to trigger the correct generation pathways.

---

# Architectural Context Block

## The "Why" — Mathematical Justification

In a standard linear layer transformation, the output is defined by:

$$y = Wx + b$$

where:

- $W$ — the dense weight matrix
- $x$ — the input token embedding
- $b$ — the bias vector

Modifying $W$ fundamentally changes how features are extracted and combined. Modifying **only** $b$ acts as a **data-independent, translation-invariant offset** in the activation space.

By shifting $b$, we change the baseline propensity of specific neurons to fire (or pass through non-linearities like GELU/Swish), which is mathematically sufficient to **align a model to a new task format** if the requisite representations already exist in $W$.

---

## VRAM & Compute Impact

### VRAM

Exceptionally low. A **7B parameter model** typically requires 16 GB VRAM just for standard full-parameter optimizer states (in FP16). BitFit reduces trainable parameters to **5–10 million**. Optimizer state memory drops to **megabytes**.

### Compute

- **Forward passes** operate at standard baseline speeds
- **Backward passes** are accelerated because the framework bypasses gradient calculations for the massive $W$ matrices, propagating errors only to the $b$ vectors

### Storage

Unlike LoRA, there is **no need for a complex adapter-merging phase** during deployment.

---

## Architectural Trade-offs

### ✅ Pros

- **Maximum Parameter Efficiency:** Often outperforming LoRA on smaller models
- **Zero Added Inference Latency:** No adapter overhead at serving time
- **Prevents Catastrophic Forgetting:** Pre-trained knowledge remains fully intact

### ❌ Cons

- **Architectural Dependency:** Modern LLM architectures (e.g., Llama-1/2/3, Mistral, Gemma) explicitly **remove bias terms** from their Linear layers to optimize kernel efficiency. BitFit is **physically impossible** on these architectures without manual structural modifications. It requires models that natively use biases, such as:
  - `OPT`
  - `BLOOM`
  - `GPT-2`

- **Expressivity Limits:** Struggles to inject complex new factual knowledge compared to higher-rank LoRA or full fine-tuning.




# Production-Grade Code / Configuration

The following implementation is designed for a **Google Colab (T4 GPU)** environment. It bypasses standard PEFT libraries (which often lack native BitFit wrappers) by manipulating **PyTorch's `requires_grad` computational graph** directly.

We utilize `facebook/opt-1.3b` because the **OPT architecture** natively includes bias terms in its attention and MLP layers, making it a **perfect candidate** for this technique.

## Environment Setup

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate datasets bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig

In [ ]:
# 1. Configuration & Hyperparameters
MODEL_ID = "facebook/opt-1.3b"
DATASET_ID = "timdettmers/openassistant-guanaco"
# T4 GPUs support float16 optimally. Bfloat16 is better for Ampere+ architectures.
TORCH_DTYPE = torch.float16

# 2. Load Tokenizer & Model (Hardware-Aware)
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=TORCH_DTYPE,
    attn_implementation="sdpa",
)

# Enable gradient checkpointing to save VRAM on the T4, even with sparse updates
model.gradient_checkpointing_enable()

# 3. The BitFit Reparameterization Logic
print("Applying BitFit Reparameterization...")
trainable_params = 0
all_params = 0

for name, param in model.named_parameters():
    all_params += param.numel()
    # Core BitFit Logic: Unfreeze ONLY if the parameter is a bias term
    if "bias" in name:
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False

print(f"Total Parameters: {all_params:,}")
print(f"Trainable Parameters (Biases Only): {trainable_params:,} ({100 * trainable_params / all_params:.4f}%)")

## Data Preparation

In [ ]:
# 4. Load & Prepare Dataset
# Using the "train" split. The dataset provides a 'text' column formatted as a conversation.
raw_dataset = load_dataset(DATASET_ID, split="train")
dataset = raw_dataset.shuffle(seed=42).select(range(500))

## Model Training

In [ ]:
training_args = SFTConfig(
    output_dir="./bitfit",
    run_name="run_bitfit",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    max_length=512,
    truncation_mode="keep_start",
    packing=False,
    completion_only_loss=True,

    optim="paged_adamw_8bit",
    learning_rate=2e-3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.001,
    max_grad_norm=0.3,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
    torch_empty_cache_steps=25,

    max_steps=200,
    num_train_epochs=3,

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    report_to="none",

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    dataset_num_proc=2,
    dataset_kwargs={
        "add_special_tokens": False,
        "skip_prepare_dataset": False,
    },
    shuffle_dataset=True,

    seed=42,
    data_seed=42,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
)

# TRAINER INITIALIZATION
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

print("[Success] fine-tuning complete. Saving adapter weights...")
trainer.model.save_pretrained("./peft_bitfit_adapter")

### To download fine-tuned model

In [ ]:
import shutil
from google.colab import files

# Name of the folder you want to download
folder_to_zip = './peft_bitfit_adapter'
# Name of the resulting zip file
output_filename = 'peft_bitfit_adapter.zip'

# Create the zip archive
shutil.make_archive('peft_bitfit_adapter', 'zip', folder_to_zip)

# Download the file to your machine
# files.download(output_filename)

In [ ]:
# ---
# To Save Model to Google Drive
# ---
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the destination path in your Drive
destination_folder = '/content/drive/MyDrive/colab_models'
os.makedirs(destination_folder, exist_ok=True)

source_path = '/content/peft_bitfit_adapter.zip'
destination_path = os.path.join(destination_folder, 'peft_bitfit_adapter.zip')

# 3. Copy the file
print(f"Copying {source_path} to {destination_path}...")
shutil.copy(source_path, destination_path)
print("Done! You can now find the model in your Google Drive under 'colab_models'.")

In [ ]:
print("Extracting and saving sparse BitFit weights...")

# 1. Pull the updated state dict containing all weights from the model
current_state_dict = model.state_dict()

# 2. Filter out everything except the bias terms
bitfit_state_dict = {k: v for k, v in current_state_dict.items() if "bias" in k}

# 3. Save strictly the biases to your local file explorer
torch.save(bitfit_state_dict, "bitfit_weights.pt")

print("Saved 'bitfit_weights.pt' successfully! Check your Colab file explorer.")

# Model Usage

## Option A: Loading the Full Model Directory

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file

# 1. Configuration
BASE_MODEL_ID = "facebook/opt-1.3b"
TUNED_DIR = "./peft_bitfit_adapter"
TORCH_DTYPE = torch.float16

print(f"Initializing Base Engine: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    attn_implementation="sdpa"
)

# 2. Dual Inference Orchestrator
def evaluate_models(prompt: str, max_new_tokens: int = 128, temperature: float = 0.7):
    """
    Evaluates both base and tuned configurations sequentially
    to preserve T4 VRAM allocations.
    """
    # Standard format used during training
    formatted_prompt = f"### Human: {prompt}\n### Assistant:"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": temperature,
        "top_p": 0.9,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id
    }

    # --- PHASE 1: Base Model Generation ---
    model.eval()
    with torch.no_grad():
        base_outputs = model.generate(**generation_kwargs)

    base_tokens = base_outputs[0][inputs.input_ids.shape[1]:]
    base_response = tokenizer.decode(base_tokens, skip_special_tokens=True).strip()

    # --- PHASE 2: Dynamic BitFit Injection ---
    safetensors_path = os.path.join(TUNED_DIR, "model.safetensors")
    bin_path = os.path.join(TUNED_DIR, "pytorch_model.bin")

    if os.path.exists(safetensors_path):
        tuned_state_dict = load_file(safetensors_path)
    elif os.path.exists(bin_path):
        tuned_state_dict = torch.load(bin_path, map_location=model.device, weights_only=True)
    else:
        raise FileNotFoundError("Target BitFit weights could not be located.")

    # Apply the delta translation layers (strict=False preserves base W matrices)
    model.load_state_dict(tuned_state_dict, strict=False)

    # --- PHASE 3: Tuned Model Generation ---
    model.eval()
    with torch.no_grad():
        tuned_outputs = model.generate(**generation_kwargs)

    tuned_tokens = tuned_outputs[0][inputs.input_ids.shape[1]:]
    tuned_response = tokenizer.decode(tuned_tokens, skip_special_tokens=True).strip()

    # --- Print Structured Analysis Matrix ---
    print("=" * 80)
    print(f"PROMPT: {prompt}")
    print("=" * 80)
    print(f" [BASE MODEL RESPONSE]\n{base_response}")
    print("-" * 80)
    print(f" [BITFIT TUNED RESPONSE]\n{tuned_response}")
    print("=" * 80)

# 3. Test Cases
evaluate_models("Write a short professional email explaining a delayed delivery.")
evaluate_models("What are the core benefits of exercising every morning?")

## Option B: Loading Sparse BitFit Bias Weights (Ultra-Efficient)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Configuration
BASE_MODEL_ID = "facebook/opt-1.3b"
BIAS_WEIGHTS_PATH = "bitfit_weights.pt"
TORCH_DTYPE = torch.float16

# 2. Load the Baseline Infrastructure
print(f"Loading Base Architecture: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    attn_implementation="sdpa"
)

# 3. Inject Tuned Bias Weights Dynamically
print(f"Injecting sparse BitFit weights from {BIAS_WEIGHTS_PATH}...")
# weights_only=True ensures a secure pickle load (standard practice in modern PyTorch)
bitfit_state_dict = torch.load(BIAS_WEIGHTS_PATH, map_location=model.device, weights_only=True)

# CRITICAL: strict=False tells the model to keep its base linear layers intact while updating only the matching bias keys found in the state dict.
model.load_state_dict(bitfit_state_dict, strict=False)
model.eval()
print("Model ready for inference!")

# 4. Inference Orchestrator
def generate_response(prompt: str, max_new_tokens: int = 128):
    formatted_prompt = f"### Human: {prompt}\n### Assistant:"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# 5. Execution
test_prompt = "Give me a quick 3-step recipe to make scrambled eggs."
print(f"\nPrompt: {test_prompt}\n")
response = generate_response(test_prompt)
print(f"Response:\n{response}")